# Chapter 03D - Should the writer have to rely on one note?

[03C](03C-fixed-windows-to-recurrent-language-models.ipynb) continued a stream of words.
Now change the job: read a complete source, then write a separate answer.
03A gave us a quick preview of that reader/writer arrangement. Here we'll take our time.

The first reader passes only its final note to the writer. We'll call that note the **summary
card**: same running-note idea, but only the final version crosses the handoff.
The second writer can also look back at the reader's earlier notes. Does that help on new examples?

Both experiments keep their own matching batch schedules and gradient clipping.
At the end, a small wrapper shows how this reader/writer can also use our shared training loop.
The wrapper doesn't teach the trainer to read or write; those jobs stay inside the model.

**Code-only view:** [Python companion](03D-recurrent-encoder-decoder-to-attention.py).
**Next:** [Stage 04](../../04-transformers/README.md) keeps the reader/writer roles but builds them with Transformer blocks.

# One reader card, or a writer that can look back?

**Guiding question:** When the answer is a different sequence, does the writer need every reader
note, or is one learned summary enough?

You can already continue a stream one token at a time. Now read the whole input before writing.
Let one component read it and another write the answer. The reader is the **encoder**.
The writer is the **decoder**. These are roles, not names for a particular internal layer.
Predicting a separate ordered answer is a **sequence-to-sequence** task.
The [03A preview, formerly 02C](03A-training-skeleton-and-word-objectives.ipynb)
introduced those jobs. Here you'll test the handoff on examples kept out of training.

Why leave the color sentences? They ask us to continue one stream, not write a separate answer.
Reverse `[3, 1, 4, 1]` into `[1, 4, 1, 3]` instead. Every needed source address is known.
This is [Stage 04's routing microscope](../../04-transformers/05-encoder-decoder-and-cross-attention-theory.md).
It avoids a second text tokenizer while the task changes.

| Attempt | Complaint | Evidence |
|---|---|---|
| Correct previous digits in practice | Generation lacks those answers | Alignment table and wrong-guess trace |
| One recurrent summary | Writer cannot reopen notes | Accuracy by source length |
| Writer asks every source note | Reading is still sequential | Same split, measured map, generation |

Prerequisites: tensor training, running notes, and token-row lookups. All state is rebuilt here.
No downloads or checkpoints. Small CPU tensors, one thread, two bounded training runs.
[Handwritten companion](03D-recurrent-encoder-decoder-to-attention-theory.md).

## 1. Read one sequence, write another

**Predict:** If a reader notices word order, does that mean its final note can fit every detail?

![The reader compresses an ordered source into one learned card. The writer cannot reopen earlier source positions.](images/03-evo-04-one-card-bottleneck.png)
*The reader can follow the whole source yet hand the writer only its final summary card.*

```text
reader: 3 -> note -> 1 -> note -> 4 -> note -> 1 -> final card
writer: final card + start -> 1 -> 4 -> 1 -> 3 -> end
```

This final card isn't an average of word rows. The reader really read them in order and
rewrote its note along the way. But however long the source gets, the final note has the
same number of entries. A short source may fit just fine. Let's measure instead of assuming
the one-card model must always fail.

## 2. Practice and generation give different help

**Predict:** Should the writer receive the current answer or the previous correct answer?
During practice, supply the correct **previous** digit. This is **teacher forcing**.
`start` is a beginning marker; `end` is a stopping answer. The current answer never appears as
its own input.

| Write step | Practice input | Expected answer |
|---|---|---|
| First | start | 1 |
| Second | 1 | 4 |
| Third | 4 | 1 |
| Fourth | 1 | 3 |
| Fifth | 3 | end |

Generation cannot borrow correct previous answers. It feeds its own guesses back.
If the first guess is 4 instead of 1, the second input becomes 4 instead of 1.
That changes the next computation. It does not prove every error must cascade.

![Practice supplies the correct previous answer. Generation feeds back its own guess, changing the next input.](images/03-evo-06-practice-vs-generation.png)
*Practice and generation use different previous answers.*


In [ ]:
import random
import sys
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
random.seed(17); np.random.seed(17); torch.manual_seed(17)
torch.set_num_threads(1)

# VS Code may start the kernel at the repository root, so find this chapter's folder.
CHAPTER_DIR = next(
    folder
    for folder in [
        Path(globals().get("__vsc_ipynb_file__", "")).resolve().parent,
        Path.cwd(),
        Path.cwd() / "learning" / "03-language-model-foundations" / "03-language-models",
    ]
    if (folder / "language_model.py").exists()
)
if str(CHAPTER_DIR) not in sys.path:
    sys.path.insert(0, str(CHAPTER_DIR))
started = time.perf_counter()

**Your turn:** Change only `first_guess`. Trace the next input before the training loop exists.

In [ ]:
first_guess = 4  # CHANGE THIS: 1 is the correct first answer
correct_target = [1, 4, 1, 3, "end"]
practice_inputs = ["start"] + correct_target[:-1]
print("Practice input -> answer:", list(zip(practice_inputs, correct_target)))
print("Generation: start -> guessed", first_guess, "-> next input", first_guess)
print("Correct previous input at step two:", practice_inputs[1])
print("Takeaway: a wrong guess changes the next input, not the stored source.")

**Take away the answer key and the inputs change.** Practice gets the correct previous
answer; generation gets whatever the model last guessed.
This mismatch is called **exposure bias**. Teacher forcing is a training method, not an
architecture requirement. It also applies to models that continue one stream.
Report both conditions rather than calling practice accuracy generation quality.

## 3. Give both models exactly the same examples

**Predict:** Will the one-card reader fail on every short source? Do not assume it.
Digits 1 through 8 are token IDs: addresses for trainable rows, not number magnitudes.
ID 0 is reserved for padding, 9 means start, and 10 means end.
An **embedding** is the trainable row retrieved by one address.
Each batch uses one source length, so it needs no padding. Both models train on lengths 2 through 8
and use the same held-out rows at each length. This tests unseen sequences at trained lengths,
not extrapolation to longer lengths. The small length-2 space limits distinct rows.

In [ ]:
PAD,BOS,EOS,V=0,9,10,11
rng=random.Random(31)
train,test={},{}
for length in range(2,9):
    unique=set()
    count=min(1200,8**length-40)
    while len(unique)<count+40:
        unique.add(tuple(rng.randint(1,8) for _ in range(length)))
    rows=list(sorted(unique)); rng.shuffle(rows)
    # One length per bucket -> ID rectangles [N_train,L] and [40,L]; no blank slots are needed.
    train[length]=torch.tensor(rows[:count]); test[length]=torch.tensor(rows[count:])

In [ ]:
for length in train:
    training_rows = {tuple(r) for r in train[length].tolist()}
    testing_rows = {tuple(r) for r in test[length].tolist()}
    assert training_rows.isdisjoint(testing_rows)
    print(f"Length {length}: {len(training_rows)} train, {len(testing_rows)} held out")
print("Takeaway: both models receive the same split; no test source appears in training.")

**Your turn:** Change only `example_length` to inspect another length bucket before fitting.

In [ ]:
example_length = 4  # CHANGE THIS: 2 through 8
example = test[example_length][0].tolist()
print("source:", example, "target:", list(reversed(example)) + [EOS])

**Before we compare:** both models get the same examples. They won't have the same number of
weights, so we'll report that difference too.

## 4. Build the reader and writer

**Predict:** What changes if the writer keeps the final summary card but can also reopen the
reader's earlier notes?

Our reader uses a **gated recurrent unit (GRU)**, another running-note design with learned
controls over what to keep and rewrite. You don't need its gate formulas for this comparison.
It really reads in order; it isn't averaging word rows. Both writers start with its final note.
The first writer gets no other way to consult the source.

The second writer can look back. At each writing step, it forms a **query**: a vector playing
the role of "what detail would help me now?" That is an analogy, not a literal question
written in words. Compare it with each saved source note, give better matches more weight,
and blend the notes using those weights. `softmax` makes the weights positive and sum to one.

That fresh blend helps the writer update its state and score the next token. This is
**attention**: a new look at the source for each writing step, rather than one summary
expected to serve every step.

![A recurrent writer looks back at all source states. End-to-start is the expected reversal route, not a fabricated measured map.](images/03-evo-05-writer-looks-back.png)
*Each writing step can ask all reader notes for a useful detail.*


In [ ]:
class ReaderWriter(nn.Module):
    def __init__(self,look_back):
        super().__init__(); self.look_back=look_back
        self.embedding=nn.Embedding(V,24)
        self.reader=nn.GRU(24,64,batch_first=True)
        self.writer=nn.GRUCell(24+(64 if look_back else 0),64)
        self.query=nn.Linear(64,64,bias=False) if look_back else None
        self.head=nn.Linear(128 if look_back else 64,V)
    def read(self,source):
        # Source IDs [B,L] -> rows [B,L,24] -> saved notes [B,L,64].
        # Final state [1,B,64] -> [B,64] drops the one-layer axis, not a source position.
        memory,last=self.reader(self.embedding(source)); return memory,last[0]
    def step(self,previous,state,memory):
        if self.look_back:
            # Project writer state [B,64] -> query [B,64]; unsqueeze -> [B,64,1].
            # Batched dot products [B,L,64] @ [B,64,1] -> [B,L,1] -> [B,L]; divide by sqrt(64) to tame score scale.
            scores=torch.bmm(memory,self.query(state).unsqueeze(2)).squeeze(2)/8
            # Normalize over source positions: [B,L] -> [B,L], one unit-sum reading budget per example.
            weights=scores.softmax(-1)
            # [B,L] -> [B,1,L] @ notes [B,L,64] -> [B,1,64] -> [B,64], a weighted source blend.
            context=torch.bmm(weights.unsqueeze(1),memory).squeeze(1)
            # Previous IDs [B] -> [B,24]; append context features [B,64] -> [B,88] for the next writer update.
            state=self.writer(torch.cat([self.embedding(previous),context],-1),state)
            # Join state/context features [B,64] + [B,64] -> [B,128]; project 128 -> V scores [B,V].
            logits=self.head(torch.cat([state,context],-1))
        else:
            # Without look-back, [B] -> [B,24] feeds the writer; its state [B,64] alone projects -> [B,V].
            state=self.writer(self.embedding(previous),state); logits=self.head(state); weights=None
        return logits,state,weights
    def forward(self,source,teacher=True):
        memory,state=self.read(source); previous=torch.full((len(source),),BOS)
        # Reverse only time [B,L] -> [B,L], then append end IDs [B,1] -> answers [B,L+1].
        target=torch.cat([source.flip(1),torch.full((len(source),1),EOS)],1) if teacher else None
        scores=[]; maps=[]
        for t in range(source.shape[1] + 1):
            logits,state,weights=self.step(previous,state,memory); scores.append(logits)
            if weights is not None: maps.append(weights)
            # Select one correct answer or one guess per row: [B,L+1] or [B,V] -> previous IDs [B].
            previous=target[:,t] if teacher else logits.argmax(-1)
        # Stack adds writer time: L+1 score arrays [B,V] -> [B,L+1,V]; reading maps [B,L] -> [B,L+1,L].
        return torch.stack(scores,1), torch.stack(maps,1) if maps else None

**Follow the notes:** `read` gives us the note saved after every source token and the final
note. `step` takes the previous answer token and updates the writer's note. If look-back is on,
it first compares the query with all saved source notes and makes a weighted blend for the
writer and output layer. Dividing the match scores by 8 keeps their scale moderate for width 64.
The last method repeats this writing step. Practice supplies the correct previous answer;
generation supplies the model's own last guess.

Unlike CBOW's one average, this blend can change at every writing step. The saved notes
stay separate until the writer compares them. Both reader and writer still update sequentially.
If you later mix lengths in a batch, keep blank source slots out of attention and blank answer
slots out of grading. Those are two different jobs for masks. Our equal-length batches don't
need either kind of padding mask.

In [ ]:
models, histories = {}, {}
for look_back in [False, True]:
    torch.manual_seed(31)
    model = ReaderWriter(look_back)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.004)
    schedule = random.Random(99)
    batch_rng = torch.Generator().manual_seed(99)
    history = []
    for update in range(1000):
        length = schedule.randint(2, 8)
        rows = train[length]
        source = rows[torch.randint(len(rows), (64,), generator=batch_rng)]
        # Reverse [64,L] and append [64,1] end IDs -> [64,L+1], including the end prediction in training.
        target = torch.cat([source.flip(1), torch.full((64, 1), EOS)], 1)
        logits, _ = model(source, teacher=True)
        # Scores [64,L+1,V] -> [64*(L+1),V], targets -> [64*(L+1)]; merge matching batch/time axes only.
        loss = F.cross_entropy(logits.flatten(0, 1), target.flatten())
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        history.append(loss.item())
    model.eval()
    models[look_back], histories[look_back] = model, history
    print("look back:", look_back, "loss", round(history[0], 4), "->", round(history[-1], 4),
          "parameters", sum(p.numel() for p in model.parameters()))

**Keep the comparison honest:** restart the batch schedule and optimizer settings for each run.
Both models get the same examples in the same order, including end as an answer to learn.
The loss trains both reader and writer. The models still have different numbers of weights;
one run can't tell us how much of any difference comes from access, size, or training luck.

**Your turn:** Change only `look_back_to_inspect`. Both printed models still have a recurrent reader.

In [ ]:
look_back_to_inspect = False  # CHANGE THIS: True
print(models[look_back_to_inspect])

**Practice went better. Can it write the answer without help?**

## 5. Grade complete answers, by source length

**Predict:** Which model will reverse more held-out rows? Will practice overstate generation?
**Sequence accuracy** requires every answer token, including end, to be correct.
One wrong token fails the whole row. Practice uses correct previous answers; free-running mode
uses its own choices. Generation begins at start and has no target inputs.
We allow source length plus one steps, a safety budget including end; source length is available
at inference. End must appear exactly after the reversed digits. Earlier end fails the row.
Continuing the budget after end cannot rescue a wrong row's exact score.

Both conditions can have equal exact accuracy: after the first wrong guess, the row already fails
either score. Equality would not mean their later histories are the same.

In [ ]:
results = []
with torch.no_grad():
    for look_back, model in models.items():
        for length, source in test.items():
            # The held-out answers use the same layout: reverse [B,L], append end [B,1] -> [B,L+1].
            target = torch.cat([source.flip(1), torch.full((len(source), 1), EOS)], 1)
            tf_scores, _ = model(source, teacher=True)
            free_scores, _ = model(source, teacher=False)
            # Choose IDs [B,L+1,V] -> [B,L+1]; all(time) -> [B] whole-answer checks, mean -> scalar accuracy.
            # The token scores below instead average every [B,L+1] check, so partial answers still count there.
            tf_ok = (tf_scores.argmax(-1) == target).all(1).float().mean().item()
            free_ok = (free_scores.argmax(-1) == target).all(1).float().mean().item()
            tf_token = (tf_scores.argmax(-1) == target).float().mean().item()
            free_token = (free_scores.argmax(-1) == target).float().mean().item()
            results.append((look_back, length, tf_ok, free_ok, tf_token, free_token))
print("look-back length practice-sequence free-sequence practice-token free-token")
for row in results:
    print(row[0], row[1], *[f"{value:.1%}" for value in row[2:]])
# Seven per-length scores -> one equally weighted mean; this is not a pooled token accuracy.
fixed_mean = np.mean([r[3] for r in results if not r[0]])
attention_mean = np.mean([r[3] for r in results if r[0]])
print(f"Mean over lengths, free-running: one card {fixed_mean:.1%}; look-back {attention_mean:.1%}")
if attention_mean > fixed_mean:
    print("Takeaway: look-back improved this held-out run; this is not a universal ranking.")
else:
    print("Takeaway: look-back did not improve this run; access exists but payoff was not established.")
fig, ax = plt.subplots(figsize=(6, 3))
for mode, label, color in [(False, "One learned card", "#ef8a23"), (True, "Writer looks back", "#2c7fb8")]:
    rows = [r for r in results if r[0] == mode]
    ax.plot([r[1] for r in rows], [r[3] for r in rows], "o-", label=label, color=color)
ax.set(xlabel="Source length", ylabel="Complete free-running accuracy", ylim=(0, 1.05))
ax.legend(); plt.show()

The look-back model also has more weights, so this isn't attention alone.

**Read the two scores separately:** test each length group twice, without changing weights.
Whole-answer accuracy asks whether every token was right. Token accuracy shows what happened
inside answers that were partly wrong. Two runs can tie on whole answers yet make different
later mistakes. The plot shows only measured generation scores, without the answer key.

**Your turn:** Change only the first digit in `source_to_reverse`. Generate without seeing the answer.
This probe is not automatically independent of training data.

In [ ]:
source_to_reverse = [3, 1, 4, 1]  # CHANGE THIS: first digit only, within 1..8
source = torch.tensor([source_to_reverse])
with torch.no_grad():
    for mode, model in models.items():
        scores, _ = model(source, teacher=False)
        print("look back", mode, "generated IDs:", scores.argmax(-1)[0].tolist())
print("Expected:", list(reversed(source_to_reverse)) + [EOS])

**We know whether the answer was right. Where did the writer look?**

## 6. Inspect the route without declaring victory from a picture

**Predict:** For `[3, 1, 4, 1]`, should reversal read source positions from last to first?
Two `1` tokens make content alone an ambiguous address. Recurrent reader states contain ordered
context. The learned map need not be a perfect anti-diagonal, a line running from upper right to
lower left. This heatmap comes from real free-running query weights, not hand-drawn target routing.

In [ ]:
with torch.no_grad():
    probe = torch.tensor([[3, 1, 4, 1]])
    logits, weights = models[True](probe, teacher=False)
    generated = logits.argmax(-1)[0].tolist()
    # Choose one example: [1,5,4] -> [5,4], writer steps as rows and source positions as columns.
    heat = weights[0].numpy()
fig, ax = plt.subplots(figsize=(6, 3))
image = ax.imshow(heat, vmin=0, vmax=1, cmap="Blues", aspect="auto")
ax.set_xticks(range(4), ["3 @ 0", "1 @ 1", "4 @ 2", "1 @ 3"])
ax.set_yticks(range(5), [f"step {i}: chose {digit}" for i, digit in enumerate(generated)])
ax.set(xlabel="Reader source position", ylabel="Writer step", title="Measured weights: diagnostic, not proof")
fig.colorbar(image, ax=ax, label="Read weight"); plt.show()
# Sum only source columns [5,4] -> [5] to check one reading budget at each writer step.
print("Row sums:", heat.sum(1).round(6).tolist())
print("Most-read source positions:", heat.argmax(1).tolist())
print("Expected digit addresses:", [3, 2, 1, 0], "; end has no single source address")
print("Takeaway: inspect routing beside generated answers; the picture alone cannot prove correctness.")

**Your turn:** Change only `writing_step`. Does its most-read source address match the reversal address?

In [ ]:
writing_step = 0  # CHANGE THIS: 1, 2, or 3
print("Most-read:", int(heat[writing_step].argmax()), "expected:", 3 - writing_step)

**A plausible picture helps. It still isn't the whole explanation.**

## 7. What did we find out - and what's still missing?

| Question | Evidence |
|---|---|
| Was the one-card model recurrent? | Learned GRU final state alone initializes its writer |
| Was look-back real? | Writer queries all retained source states at each output step |
| Did it help? | Per-length and mean free-running scores with conditional takeaway |
| Is practice generation? | Both exact-sequence and token scores printed separately |
| Does the map prove the answer? | No; it diagnoses one route beside behavior |

| Teaching mechanism | Production counterpart | What changes | What stays |
|---|---|---|---|
| One reader card | Recurrent fixed-summary model | Larger state and data | No direct source addresses |
| Writer asks reader states | Recurrent attention | Scoring choices and masks | New source blend per step |
| Correct previous practice token | Teacher-forced training | Vocabulary and batch size | Generation uses own history |
| Recurrent reader/writer | Transformer encoder-decoder | Self-attention replaces recurrence | Separate reading/writing roles |

In [ ]:
print({"fixed_summary_mean_free_sequence_accuracy": round(float(fixed_mean), 6),
       "attention_mean_free_sequence_accuracy": round(float(attention_mean), 6),
       "held_out_sequences_per_length": len(test[2]), "source_lengths": list(test)})
print(f"Elapsed CPU wall time: {time.perf_counter() - started:.1f}s")

### What we tried, explained, and left for later

- **What we built and checked:** practice table, wrong-guess input trace, disjoint split, learned GRU
  fixed-summary reader/writer, query-based recurrent source attention, matched batch schedule,
  parameter counts, per-length practice/free-running sequence and token accuracy, generated probes,
  and measured source-routing heatmap.
- **What we explained:** teacher forcing and exposure bias, one-card limitation,
  source masks versus loss masks, gates versus source access, repeated-digit ambiguity,
  bidirectional encoders (a complete source may use both directions, unlike generated history),
  and attention predating Transformers.
- **What we're leaving for later:** LSTM gate internals (existing memory lesson), beam search
  (keep several candidate answers), scheduled sampling (vary practice history), extrapolation,
  multi-seed rankings, production translation, and [Transformer internals](../../04-transformers/README.md).

**Watch out for these traps:** do not substitute mean pooling for a learned recurrent summary;
do not feed the current answer as its own input; do not grade or read blank slots;
do not call practice generation; do not mistake a pretty map for proof.

Attention predates Transformers. LSTM can help retain information. It doesn't remove the one-card handoff.
A gated reader can still hand the writer only one card.

**The question we're left with:** Looking back changes source access, but recurrent reading remains sequential.
[Stage 04](../../04-transformers/README.md) asks: **what if we drop recurrence and let attention do all the reading?**

## 8. Can this reader/writer use the shared training loop too?

So far, we wrote out the practice schedule to keep the two experiments comparable.
That doesn't mean an encoder-decoder needs its own training framework.
It just returns an extra result that our common loop doesn't need.

`ReaderWriter` returns `(logits, maps)`: word scores plus the attention picture.
Wrap it in a small model whose `forward` returns just the scores.
PyTorch registers the wrapped model as a **submodule**, so its weights are still included.
The correct answers already contain the reversed digits followed by end, so no target shift
is needed here. Teacher forcing stays inside `ReaderWriter`.

There is a limit to this wrapper: the reversal model constructs its correct practice prefix
from the source. This isn't a general interface for arbitrary translation pairs.

```text
adapter.fit_language_model(...)       inherited practice loop
-> adapter.training_loss(...)         inherited loss calculation
-> adapter.forward(source, targets)   get ReaderWriter's practice scores
-> ReaderWriter.read -> ReaderWriter.step
-> cross-entropy -> backward          reaches embedding + reader + writer
```

We'll take one verification step on a **deep copy**, not the models we just evaluated.
That keeps their scores and attention maps unchanged. We're checking that the pieces connect,
not running another quality contest or replacing the original clipped training schedule.

In [ ]:
from copy import deepcopy
from language_model import TrainableLanguageModel
from torch.utils.data import DataLoader, TensorDataset


class ReaderWriterTrainingModel(TrainableLanguageModel):
    def __init__(self, reader_writer):
        super().__init__()
        self.reader_writer = reader_writer

    def forward(self, source_ids, target_ids=None):
        # Keep scores [B,L+1,V] for the shared loss; reading maps are a diagnostic, not a second prediction target.
        logits, _ = self.reader_writer(source_ids, teacher=True)
        return logits


adapter = ReaderWriterTrainingModel(deepcopy(models[False]))
adapter_source = train[4][:8]
# Reverse source [8,4] and append end IDs [8,1] -> [8,5], aligned with the writer's five output steps.
adapter_target = torch.cat(
    [adapter_source.flip(1), torch.full((len(adapter_source), 1), EOS)], dim=1,
)
adapter_loader = DataLoader(TensorDataset(adapter_source, adapter_target), batch_size=8)
adapter_optimizer = torch.optim.Adam(adapter.parameters(), lr=0.001)
original_embedding = models[False].embedding.weight.detach().clone()
adapter_before = adapter.reader_writer.embedding.weight.detach().clone()

assert ReaderWriterTrainingModel.fit_language_model is TrainableLanguageModel.fit_language_model
adapter_losses = adapter.fit_language_model(
    adapter_loader, nn.CrossEntropyLoss(), adapter_optimizer, epochs=1,
)
assert torch.equal(models[False].embedding.weight.detach(), original_embedding)
adapter_change = (adapter.reader_writer.embedding.weight.detach() - adapter_before).abs().max().item()
assert adapter_change > 0
print("Inherited encoder-decoder training loss:", adapter_losses[0])
print("Embedding updated through reader and writer; original experiment unchanged:", adapter_change)

**Follow the call:** the shared loop reaches the wrapper's `forward`, which calls the same
reader/writer we already understand. Only the attention map is left out of the loss calculation.
Feedback still reaches the embedding, reader, and writer. The weight-equality check confirms
we haven't changed the original model while testing its copy.

**What we checked:** inherited training works here, and the embedding weights update.
**What we explained:** how a small, task-specific wrapper adapts the returned results.
**What we left for later:** a trainer for variable-length batches, clipping hooks in the shared
loop, and training spread across devices.

Look back at the route: fetch word rows, teach them with guessing games, give words ordered
seats, carry a running note, then let a writer reopen earlier source notes.
**We fixed source access. The reader and writer still work one step at a time.**

[Stage 04](../../04-transformers/README.md) picks up there with attention, position, masks,
and complete Transformer blocks. This is a learning route, not a family tree: CBOW isn't
a Transformer component, LSTM isn't a compulsory stop, and adding attention alone doesn't
make a model a Transformer.